In [7]:
"""
Tarea_2.py
Codigo extraido de Tarea_2.pdf (Metodos de Optimizacion - Clase 2),
organizado por ejercicio. Cada ejercicio es una funcion independiente
ejercicio_N(), de modo que los nombres (f, POZOS, etc.) no se pisan
entre ejercicios.

Notas:
- Ejercicio 1: el PDF solo incluye en codigo la verificacion con brentq
  (la biseccion manual se describe con algoritmo y tabla, sin codigo).
- Ejercicio 7: es una reflexion teorica y no contiene codigo.
- Uso: python Tarea_2.py            -> ejecuta todos los ejercicios
       python Tarea_2.py 3 5        -> ejecuta solo los ejercicios 3 y 5
"""
import sys


# =====================================================================
# EJERCICIO 1 - Biseccion para minimizar f(x) = x^4 - 3x^2 + x
# (verificacion con scipy.optimize.brentq, con diagnostico completo)
# =====================================================================
def ejercicio_1():
    from scipy.optimize import brentq

    fprime = lambda x: 4*x**3 - 6*x + 1

    # Pasamos full_output=True
    x_star, info = brentq(fprime, -2, 0, xtol=1e-6, full_output=True)

    print(f"Raiz hallada: {x_star}")
    print(f"Numero de iteraciones: {info.iterations}")
    print(f"Evaluaciones de la funcion: {info.function_calls}")
    print(f"Convergio?: {info.converged}")


# =====================================================================
# EJERCICIO 2 - Comparacion de biseccion y busqueda aurea
# =====================================================================
def ejercicio_2():
    import numpy as np

    f = lambda x: x**2 - 2*x + 3
    fprime = lambda x: 2*x - 2

    def biseccion_count(fprime, a, b, tol):
        it = 0
        fa = fprime(a)
        while True:
            it += 1
            c = 0.5*(a+b)
            fc = fprime(c)
            ancho = b - a
            if abs(fc) < tol or ancho < tol:
                return it, c
            if np.sign(fc) == np.sign(fa):
                a = c
                fa = fc
            else:
                b = c

    def aurea_count(f, a, b, tol):
        PHI = (5**0.5 - 1)/2
        x1 = b - PHI*(b-a)
        x2 = a + PHI*(b-a)
        f1, f2 = f(x1), f(x2)
        it = 0
        while (b-a) > tol:
            it += 1
            if f1 < f2:
                b = x2
                x2, f2 = x1, f1
                x1 = b - PHI*(b-a)
                f1 = f(x1)
            else:
                a = x1
                x1, f1 = x2, f2
                x2 = a + PHI*(b-a)
                f2 = f(x2)
        return it, 0.5*(a+b)

    # Definicion del intervalo
    a, b = -2, 5

    # Iteracion e impresion
    tolerancias = [1e-3, 1e-6, 1e-9]

    for tol in tolerancias:
        it_bis, x_bis = biseccion_count(fprime, a, b, tol)
        it_aur, x_aur = aurea_count(f, a, b, tol)

        print(f"tol={tol:.0e} | biseccion: it={it_bis}, x*={x_bis:.8f} | "
              f"aurea: it={it_aur}, x*={x_aur:.8f}")


# =====================================================================
# EJERCICIO 3 - Grid search en 4 variables vs. 2 variables
# =====================================================================
def ejercicio_3():
    import itertools, numpy as np, timeit

    N = 20
    rango = np.linspace(-5, 5, N)

    def grid_2d_loop():
        mejor = None
        f_best = np.inf
        for punto in itertools.product(rango, repeat=2):
            val = punto[0]**2 + punto[1]**2
            if val < f_best:
                f_best = val
                mejor = punto
        return mejor, f_best

    def grid_4d():
        mejor = None
        f_best = np.inf
        for punto in itertools.product(rango, repeat=4):
            val = punto[0]**2 + punto[1]**2 + punto[2]**2 + punto[3]**2
            if val < f_best:
                f_best = val
                mejor = punto
        return mejor, f_best

    t2 = timeit.timeit(grid_2d_loop, number=20)/20
    t4 = timeit.timeit(grid_4d, number=20)/20

    print("tiempo promedio d=2 (bucle):", t2)
    print("tiempo promedio d=4 (bucle):", t4)
    print("razon tiempo:", t4/t2)


# =====================================================================
# EJERCICIO 4 - Random walk con mecanismo de reinicio
# =====================================================================
def ejercicio_4():
    import numpy as np

    # --- Reconstruccion del paisaje de 6 pozos (version del ejercicio 4) ---
    POZOS = [
        ("pozo1", -6.0,  6.0, -1.8),
        ("pozo2",  5.0,  5.5, -2.2),
        ("pozo3", -7.0, -3.0, -2.6),
        ("pozo4",  6.5, -6.0, -3.29),   # pozo global, profundidad -3.29
        ("pozo5",  0.0,  7.5, -1.5),
        ("pozo6", -3.5, -7.0, -2.0),
    ]

    def f(x, y):
        total = 0.0
        for _, cx, cy, depth in POZOS:
            r2 = (x - cx)**2 + (y - cy)**2
            total = total + depth * np.exp(-r2 / 8.0)
        return total

    # --- Codigo modificado: random walk greedy con reinicio ---
    def random_walk_greedy(f, x0, step=1.0, n_iter=300, seed=0):
        """Version ORIGINAL, sin reinicio (la del script de la Clase 2)."""
        rng = np.random.default_rng(seed)
        x = np.array(x0, dtype=float)
        fx = f(*x)
        trayectoria = [x.copy()]
        aceptados = 0
        for _ in range(n_iter):
            paso = rng.uniform(-step, step, size=2)
            x_new = x + paso
            f_new = f(*x_new)
            if f_new < fx:
                x, fx = x_new, f_new
                trayectoria.append(x.copy())
                aceptados += 1
        return x, fx, np.array(trayectoria), aceptados / n_iter

    def random_walk_greedy_restart(f, x0, step=1.0, n_iter=300, seed=0,
                                   paciencia=50, dominio=(-10, 10)):
        """Version MODIFICADA: agrega reinicio tras 'paciencia' pasos sin mejora."""
        rng = np.random.default_rng(seed)
        x = np.array(x0, dtype=float)
        fx = f(*x)
        x_best, f_best = x.copy(), fx
        trayectoria = [x.copy()]
        aceptados = 0
        reinicios = 0
        sin_mejora = 0

        for _ in range(n_iter):
            paso = rng.uniform(-step, step, size=2)
            x_new = x + paso
            f_new = f(*x_new)
            if f_new < fx:
                x, fx = x_new, f_new
                trayectoria.append(x.copy())
                aceptados += 1
                sin_mejora = 0
                if fx < f_best:
                    x_best, f_best = x.copy(), fx
            else:
                sin_mejora += 1

            # --- MECANISMO DE REINICIO ---
            if sin_mejora >= paciencia:
                x = rng.uniform(dominio[0], dominio[1], size=2)
                fx = f(*x)
                trayectoria.append(x.copy())
                sin_mejora = 0
                reinicios += 1
                if fx < f_best:
                    x_best, f_best = x.copy(), fx

        return x_best, f_best, np.array(trayectoria), aceptados / n_iter, reinicios

    # --- Experimento: punto de arranque atrapado en un minimo local ---
    x0 = np.array([-6.0, 4.0])  # cerca del pozo local, lejos del pozo global

    xg, fg, tray_g, tasa_g = random_walk_greedy(f, x0, step=1.5, n_iter=500, seed=0)
    print(f"[SIN reinicio] x*={xg.round(3)} f*={fg:.4f} "
          f"({len(tray_g)-1} pasos aceptados de 500, tasa={tasa_g*100:.1f}%)")

    xr, fr, tray_r, tasa_r, reinicios = random_walk_greedy_restart(
        f, x0, step=1.5, n_iter=500, seed=0, paciencia=50)
    print(f"[CON reinicio] x*={xr.round(3)} f*={fr:.4f} "
          f"(tasa aceptacion={tasa_r*100:.1f}%, reinicios={reinicios})")

    # --- Verificacion de robustez con multiples semillas ---
    resultados = []
    for seed in range(10):
        xg_s, fg_s, _, _ = random_walk_greedy(f, x0, step=1.5, n_iter=500, seed=seed)
        xr_s, fr_s, _, _, rein_s = random_walk_greedy_restart(
            f, x0, step=1.5, n_iter=500, seed=seed, paciencia=50)
        resultados.append((seed, fg_s, fr_s, rein_s))
        print(f"seed={seed} | f* sin reinicio={fg_s:.4f} | "
              f"f* con reinicio={fr_s:.4f} | reinicios={rein_s}")


# =====================================================================
# EJERCICIO 5 - Nelder-Mead en una funcion no convexa
# =====================================================================
def ejercicio_5():
    import numpy as np
    from scipy.optimize import minimize

    f = lambda p: (p[0]-2)**2 + (p[1]+1)**2 + 5*np.sin(p[0])*np.cos(p[1])

    # --- Localizacion del minimo global de referencia (por malla fina) ---
    xs = np.linspace(-10, 10, 800)
    ys = np.linspace(-10, 10, 800)
    X, Y = np.meshgrid(xs, ys)
    Z = (X-2)**2 + (Y+1)**2 + 5*np.sin(X)*np.cos(Y)
    idx = np.unravel_index(np.argmin(Z), Z.shape)
    print(f"Minimo de referencia (malla): x=({X[idx]:.4f}, {Y[idx]:.4f}) f={Z[idx]:.4f}")

    # --- Nelder-Mead manual ---
    def nelder_mead(f, x0, paso_inicial=1.0, tol=1e-8, max_iter=200, verbose=False):
        n = len(x0)
        simplex = [np.array(x0, dtype=float)]
        for i in range(n):
            p = np.array(x0, dtype=float)
            p[i] += paso_inicial
            simplex.append(p)

        historia = [np.array(simplex)]

        for it in range(1, max_iter + 1):
            simplex.sort(key=f)
            B, *resto, P = simplex
            fB, fP = f(B), f(P)

            if abs(fP - fB) < tol:
                break

            C = np.mean(simplex[:-1], axis=0)
            R = C + 1.0 * (C - P)
            fR = f(R)

            if fR < fB:
                E = C + 2.0 * (R - C)
                simplex[-1] = E if f(E) < fR else R
            elif fR < f(resto[-1]):
                simplex[-1] = R
            else:
                Ct = C + 0.5 * (P - C)
                if f(Ct) < fP:
                    simplex[-1] = Ct
                else:
                    simplex = [B + 0.5 * (v - B) for v in simplex]

            historia.append(np.array(simplex))

        simplex.sort(key=f)
        return simplex[0], f(simplex[0]), historia, it

    # --- Experimento con el punto de arranque original del script ---
    x0 = [4.0, 3.0]

    x_star, f_star, historia, n_it = nelder_mead(f, x0, paso_inicial=1.5, tol=1e-8)
    print(f"Manual -> x* = {x_star.round(6)} f(x*) = {f_star:.6f} ({n_it} iteraciones)")

    res = minimize(f, x0, method="Nelder-Mead", options={"xatol": 1e-8, "fatol": 1e-8})
    print(f"scipy -> x* = {res.x.round(6)} f(x*) = {res.fun:.6f} ({res.nit} iteraciones)")

    # --- Prueba con multiples puntos de partida ---
    starting_points = [[4.0, 3.0], [-3.0, -3.0], [0.0, 0.0],
                       [6.0, -6.0], [-5.0, 5.0], [2.0, 2.0]]

    for x0_ in starting_points:
        xm, fm, hist, itm = nelder_mead(f, x0_, paso_inicial=1.5, tol=1e-8)
        res_ = minimize(f, x0_, method="Nelder-Mead",
                        options={"xatol": 1e-8, "fatol": 1e-8})
        print(f"x0={x0_} | manual: x*={xm.round(3)} f*={fm:.4f} | "
              f"scipy: x*={res_.x.round(3)} f*={res_.fun:.4f}")


# =====================================================================
# EJERCICIO 6 - Error de grid search vs. evaluaciones (paisaje de 6 pozos)
# =====================================================================
def ejercicio_6():
    import numpy as np
    from scipy.optimize import minimize
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    # --- 1) Definir f y POZOS (landscape.py) ---
    # cada pozo: (profundidad, centro_x, centro_y, ancho)
    POZOS = [
        (-5.0, -6.5, 4.0, 1.3),   # el mas profundo: minimo GLOBAL
        (-4.0,  5.0, 4.0, 1.1),
        (-3.0, -5.0, -4.0, 1.1),
        (-2.0,  5.0, -5.0, 1.0),
        (-1.8,  0.0, 6.0, 1.0),
        (-1.5, -6.0, 0.0, 1.0),
    ]

    def f(x, y):
        z = 0.03 * (x**2 + y**2)
        for w, cx, cy, s in POZOS:
            z += w * np.exp(-((x - cx)**2 + (y - cy)**2) / (2 * s**2))
        return z

    def grid_search_2d(f, rango, N):
        xs = np.linspace(rango[0], rango[1], N)
        ys = np.linspace(rango[0], rango[1], N)
        X, Y = np.meshgrid(xs, ys)
        Z = f(X, Y)
        idx = np.unravel_index(np.argmin(Z), Z.shape)
        return X[idx], Y[idx], Z[idx]

    # --- 2) Calcular el minimo verdadero ANTES del bucle ---
    res_true = minimize(lambda p: f(p[0], p[1]), x0=[-6.5, 4.0], method="Nelder-Mead",
                        options={"xatol": 1e-10, "fatol": 1e-12})
    x_true, y_true = res_true.x
    f_true = res_true.fun
    print(f"Minimo verdadero: x*=({x_true:.6f},{y_true:.6f}) f*={f_true:.8f}")

    # --- 3) Barrido de N ---
    Ns = [10, 25, 50, 100]
    rango = (-10, 10)

    resultados = []
    for N in Ns:
        xg, yg, fg = grid_search_2d(f, rango, N)
        error_f = abs(fg - f_true)
        error_dist = np.hypot(xg - x_true, yg - y_true)
        evaluaciones = N**2
        resultados.append((N, evaluaciones, xg, yg, fg, error_f, error_dist))
        print(f"N={N:4d} | evals={evaluaciones:6d} | x*=({xg:7.3f},{yg:7.3f}) "
              f"| f*={fg:9.5f} | error_f={error_f:.5f} | error_dist={error_dist:.5f}")

    # --- 4) Grafica log-log: error vs. evaluaciones ---
    evaluaciones_arr = np.array([r[1] for r in resultados])   # N^2
    error_f_arr = np.array([r[5] for r in resultados])        # error_f
    error_dist_arr = np.array([r[6] for r in resultados])     # error_dist

    # Ajuste de ley de potencia: log10(error) = pendiente*log10(evals) + intercepto
    slope_f, intercept_f = np.polyfit(np.log10(evaluaciones_arr), np.log10(error_f_arr), 1)
    slope_d, intercept_d = np.polyfit(np.log10(evaluaciones_arr), np.log10(error_dist_arr), 1)

    fig, ax = plt.subplots(figsize=(7.5, 6))

    # Puntos medidos
    ax.loglog(evaluaciones_arr, error_f_arr, "o-", color="crimson", lw=2, ms=9,
              label=r"error$_f$ = $|f_{malla}-f^*|$")
    ax.loglog(evaluaciones_arr, error_dist_arr, "s-", color="navy", lw=2, ms=9,
              label=r"error$_{dist}$ = $\|x_{malla}-x^*\|$")

    # Rectas de ajuste (ley de potencia)
    fit_f = 10**intercept_f * evaluaciones_arr**slope_f
    fit_d = 10**intercept_d * evaluaciones_arr**slope_d
    ax.loglog(evaluaciones_arr, fit_f, "--", color="crimson", alpha=0.5,
              label=f"ajuste: evals$^{{{slope_f:.2f}}}$")
    ax.loglog(evaluaciones_arr, fit_d, "--", color="navy", alpha=0.5,
              label=f"ajuste: evals$^{{{slope_d:.2f}}}$")

    # Etiquetas de N en cada punto
    for N, ev, xg, yg, fg, ef, ed in resultados:
        ax.annotate(f"N={N}", (ev, ef), textcoords="offset points",
                    xytext=(6, 6), fontsize=8, color="crimson")

    ax.set_xlabel(r"Evaluaciones ($N^2$) -- escala log", fontsize=11)
    ax.set_ylabel("Error -- escala log", fontsize=11)
    ax.set_title("Grid search en paisaje de 6 pozos:\nerror vs. evaluaciones (log-log)",
                 fontsize=12)
    ax.grid(True, which="both", ls=":", alpha=0.6)
    ax.legend(fontsize=9, loc="upper right")
    fig.tight_layout()
    fig.savefig("error_vs_evaluaciones_loglog.png", dpi=150)
    print("Figura guardada: error_vs_evaluaciones_loglog.png")


# =====================================================================
# EJERCICIO 7 - Reflexion: Nelder-Mead vs. Grid Search
# (ejercicio teorico: el PDF no contiene codigo)
# =====================================================================
def ejercicio_7():
    print("Ejercicio 7: reflexion teorica, sin codigo en Tarea_2.pdf.")


EJERCICIOS = {1: ejercicio_1, 2: ejercicio_2, 3: ejercicio_3, 4: ejercicio_4,
              5: ejercicio_5, 6: ejercicio_6, 7: ejercicio_7}

if __name__ == "__main__":
    seleccion = [int(a) for a in sys.argv[1:] if a.isdigit()] or sorted(EJERCICIOS)
    for n in seleccion:
        print(f"\n{'='*20} EJERCICIO {n} {'='*20}")
        EJERCICIOS[n]()



==================== EJERCICIO 1 ====================
Raiz hallada: -1.3008395659988075
Numero de iteraciones: 10
Evaluaciones de la funcion: 11
Convergio?: True

==================== EJERCICIO 2 ====================
tol=1e-03 | biseccion: it=13, x*=1.00012207 | aurea: it=19, x*=1.00022655
tol=1e-06 | biseccion: it=22, x*=1.00000024 | aurea: it=33, x*=0.99999983
tol=1e-09 | biseccion: it=31, x*=1.00000000 | aurea: it=48, x*=1.00000002

==================== EJERCICIO 3 ====================
tiempo promedio d=2 (bucle): 0.00012046599999848695
tiempo promedio d=4 (bucle): 0.0944464222499846
razon tiempo: 784.0089506679964

==================== EJERCICIO 4 ====================
[SIN reinicio] x*=[-5.951  5.954] f*=-1.8124 (9 pasos aceptados de 500, tasa=1.8%)
[CON reinicio] x*=[ 6.43  -5.968] f*=-3.2876 (tasa aceptacion=8.8%, reinicios=5)
seed=0 | f* sin reinicio=-1.8124 | f* con reinicio=-3.2876 | reinicios=5
seed=1 | f* sin reinicio=-1.8111 | f* con reinicio=-3.2681 | reinicios=4
seed=2 |